### Импорты, seed и устройство

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
import csv
import random
from collections import OrderedDict

# Настройка отображения графиков.
%matplotlib inline
plt.style.use('ggplot')

# Создаем необходимые папки для артефактов.
ARTIFACTS_DIR = Path('artifacts')
FIGURES_DIR = ARTIFACTS_DIR / 'figures'
ARTIFACTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print(f"Артефакты будут сохраняться в: {ARTIFACTS_DIR.absolute()}")

### Фиксация seed и определение устройства.

In [ ]:
def set_seed(seed=42):
    """Фиксирует seed для воспроизводимости."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f"Seed установлен: {seed}")

SEED = 42
set_seed(SEED)

# Определяем устройство (GPU если доступно).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

### Загрузка данных и создание DataLoader'ов (2.3.2. Данные и DataLoader).

In [ ]:
# Параметры данных.
BATCH_SIZE = 64
VAL_SPLIT = 0.2

# Определяем transform для CIFAR10.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                        std=[0.2470, 0.2435, 0.2616])
])

# Загружаем тренировочную и тестовую части.
print("Загрузка CIFAR10...")
train_val_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
print("Датасет успешно загружен!")

# Воспроизводимое разбиение train/val.
dataset_size = len(train_val_dataset)
indices = list(range(dataset_size))
split = int(np.floor(VAL_SPLIT * dataset_size))

# Перемешиваем индексы с фиксированным seed.
np.random.seed(SEED)
np.random.shuffle(indices)

train_indices, val_indices = indices[split:], indices[:split]

# Создаем subsets.
train_dataset = Subset(train_val_dataset, train_indices)
val_dataset = Subset(train_val_dataset, val_indices)

print(f"Размер Train: {len(train_dataset)}")
print(f"Размер Val: {len(val_dataset)}")
print(f"Размер Test: {len(test_dataset)}")

# Создаем DataLoaders.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Sanity-check: смотрим на один batch.
data_iter = iter(train_loader)
images, labels = next(data_iter)
print(f"\n--- Sanity Check ---")
print(f"Форма батча изображений: {images.shape}")
print(f"Форма батча меток: {labels.shape}")
print(f"Минимальное значение в пикселях: {images.min():.2f}")
print(f"Максимальное значение в пикселях: {images.max():.2f}")
print(f"Уникальные метки в батче: {torch.unique(labels)}")

plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i+1)
    img = images[i].numpy().transpose(1, 2, 0)
    mean = np.array([0.4914, 0.4822, 0.4465])
    std = np.array([0.2470, 0.2435, 0.2616])
    img = std * img + mean
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    plt.title(f'Label: {labels[i].item()}')
    plt.axis('off')
plt.tight_layout()
plt.show()

# Названия классов CIFAR10
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
print(f"\nКлассы CIFAR10: {cifar10_classes}")

### Модель MLP и цикл обучения.

In [ ]:
# Модель MLP и цикл обучения (модель для CIFAR10).

class MLP(nn.Module):
    """
    Многослойный перцептрон для CIFAR10 с возможностью добавления Dropout и BatchNorm.
    Input: 3x32x32 = 3072 пикселя
    """
    def __init__(self, input_size=3072, hidden_sizes=[1024, 512, 256], num_classes=10,
                 dropout_rate=0.0, use_batchnorm=False, activation='relu'):
        super().__init__()
        self.input_size = input_size
        self.hidden_sizes = hidden_sizes
        self.num_classes = num_classes
        self.dropout_rate = dropout_rate
        self.use_batchnorm = use_batchnorm

        # Выбор функции активации
        if activation == 'relu':
            self.activation_fn = nn.ReLU()
        elif activation == 'tanh':
            self.activation_fn = nn.Tanh()
        else:
            raise ValueError("Unsupported activation")

        layers = OrderedDict()
        # Добавляем слой Flatten в начало.
        layers['flatten'] = nn.Flatten()

        prev_size = input_size
        for i, hidden_size in enumerate(hidden_sizes):
            layers[f'linear_{i}'] = nn.Linear(prev_size, hidden_size)

            if use_batchnorm:
                layers[f'batchnorm_{i}'] = nn.BatchNorm1d(hidden_size)

            layers[f'activation_{i}'] = self.activation_fn

            if dropout_rate > 0:
                layers[f'dropout_{i}'] = nn.Dropout(dropout_rate)

            prev_size = hidden_size

        layers['output'] = nn.Linear(prev_size, num_classes)

        self.network = nn.Sequential(layers)

    def forward(self, x):
        return self.network(x)

    def get_summary(self):
        """Возвращает краткое описание архитектуры для CSV."""
        bn_str = "BN" if self.use_batchnorm else "noBN"
        do_str = f"DO{self.dropout_rate}" if self.dropout_rate > 0 else "noDO"
        return f"MLP_{self.hidden_sizes}_{bn_str}_{do_str}"

In [ ]:
# Модель MLP и цикл обучения (тренировочный цикл).

def compute_accuracy(model, data_loader, device):
    """Вычисляет accuracy на предоставленном DataLoader'е."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """Обучает модель одну эпоху и возвращает средний loss."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass.
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass и оптимизация.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        
        # Считаем accuracy для train.
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def evaluate(model, val_loader, criterion, device):
    """Оценивает модель на валидационной выборке."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def run_experiment(model, train_loader, val_loader, criterion, optimizer, device,
                   num_epochs, experiment_id, early_stopping_patience=None):
    """
    Запускает эксперимент и возвращает историю и лучшую модель (если используется EarlyStopping).
    """
    history = {
        'train_loss': [],
        'train_accuracy': [],
        'val_loss': [],
        'val_accuracy': [],
        'epochs_trained': 0
    }

    best_val_acc = 0.0
    best_model_state = None
    epochs_no_improve = 0

    print(f"\nНачало эксперимента {experiment_id}")
    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)
        history['epochs_trained'] += 1

        print(f"Epoch [{epoch+1}/{num_epochs}] | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        # Проверка на улучшение для Early Stopping.
        if early_stopping_patience is not None:
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model_state = model.state_dict().copy()  # Копируем состояние.
                epochs_no_improve = 0
                print(f"  ✓ Новый лучший результат! Val Acc: {val_acc:.4f}")
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= early_stopping_patience:
                    print(f"Ранняя остановка на эпохе {epoch+1}")
                    break

    # Если early stopping не использовался, находим лучшую эпоху по val_accuracy
    if early_stopping_patience is None:
        best_epoch_idx = np.argmax(history['val_accuracy'])
        best_val_acc = history['val_accuracy'][best_epoch_idx]
        print(f"Лучшая Val Accuracy: {best_val_acc:.4f} на эпохе {best_epoch_idx+1}")

    return history, best_model_state, best_val_acc

### Эксперименты.

In [ ]:
# 3.1. Часть A: регуляризация и переобучение.

# Общие параметры для экспериментов.
criterion = nn.CrossEntropyLoss()
NUM_EPOCHS_E123 = 20
OPTIMIZER_CONFIG = {'name': 'Adam', 'lr': 0.001, 'weight_decay': 0.0}

# Словарь для хранения результатов.
results = []

print("="*60)
print("Эксперимент E1: Base Model (без Dropout, без BatchNorm)")
print("="*60)
model_e1 = MLP(hidden_sizes=[1024, 512, 256], 
               dropout_rate=0.0, 
               use_batchnorm=False).to(device)
optimizer_e1 = optim.Adam(model_e1.parameters(), lr=OPTIMIZER_CONFIG['lr'])
history_e1, _, best_val_acc_e1 = run_experiment(
    model_e1, train_loader, val_loader, criterion, optimizer_e1, device, 
    NUM_EPOCHS_E123, 'E1'
)
results.append({
    'experiment_id': 'E1',
    'model_summary': model_e1.get_summary(),
    'optimizer': 'Adam',
    'lr': OPTIMIZER_CONFIG['lr'],
    'weight_decay': 0.0,
    'epochs_trained': NUM_EPOCHS_E123,
    'best_val_accuracy': best_val_acc_e1,
    'best_val_loss': min(history_e1['val_loss']) if history_e1['val_loss'] else None
})

print("\n" + "="*60)
print("Эксперимент E2: Dropout (p=0.3)")
print("="*60)
model_e2 = MLP(hidden_sizes=[1024, 512, 256], 
               dropout_rate=0.3, 
               use_batchnorm=False).to(device)
optimizer_e2 = optim.Adam(model_e2.parameters(), lr=OPTIMIZER_CONFIG['lr'])
history_e2, _, best_val_acc_e2 = run_experiment(
    model_e2, train_loader, val_loader, criterion, optimizer_e2, device, 
    NUM_EPOCHS_E123, 'E2'
)
results.append({
    'experiment_id': 'E2',
    'model_summary': model_e2.get_summary(),
    'optimizer': 'Adam',
    'lr': OPTIMIZER_CONFIG['lr'],
    'weight_decay': 0.0,
    'epochs_trained': NUM_EPOCHS_E123,
    'best_val_accuracy': best_val_acc_e2,
    'best_val_loss': min(history_e2['val_loss']) if history_e2['val_loss'] else None
})

print("\n" + "="*60)
print("Эксперимент E3: BatchNorm")
print("="*60)
model_e3 = MLP(hidden_sizes=[1024, 512, 256], 
               dropout_rate=0.0, 
               use_batchnorm=True).to(device)
optimizer_e3 = optim.Adam(model_e3.parameters(), lr=OPTIMIZER_CONFIG['lr'])
history_e3, _, best_val_acc_e3 = run_experiment(
    model_e3, train_loader, val_loader, criterion, optimizer_e3, device, 
    NUM_EPOCHS_E123, 'E3'
)
results.append({
    'experiment_id': 'E3',
    'model_summary': model_e3.get_summary(),
    'optimizer': 'Adam',
    'lr': OPTIMIZER_CONFIG['lr'],
    'weight_decay': 0.0,
    'epochs_trained': NUM_EPOCHS_E123,
    'best_val_accuracy': best_val_acc_e3,
    'best_val_loss': min(history_e3['val_loss']) if history_e3['val_loss'] else None
})

# Выбираем лучшую модель между E2 и E3 для E4.
best_e2_e3 = max(results[-2:], key=lambda x: x['best_val_accuracy'])
best_model_class = model_e2 if best_e2_e3['experiment_id'] == 'E2' else model_e3
print(f"\n🔍 Лучшая модель между E2 и E3: {best_e2_e3['experiment_id']} "
      f"с Val Acc = {best_e2_e3['best_val_accuracy']:.4f}")

print("\n" + "="*60)
print(f"Эксперимент E4: Early Stopping на базе {best_e2_e3['experiment_id']}")
print("="*60)
model_e4 = MLP(hidden_sizes=[1024, 512, 256],
               dropout_rate=(0.3 if best_e2_e3['experiment_id'] == 'E2' else 0.0),
               use_batchnorm=(best_e2_e3['experiment_id'] == 'E3')).to(device)
optimizer_e4 = optim.Adam(model_e4.parameters(), lr=OPTIMIZER_CONFIG['lr'])
history_e4, best_model_state_e4, best_val_acc_e4 = run_experiment(
    model_e4, train_loader, val_loader, criterion, optimizer_e4, device,
    num_epochs=40,  # Больше эпох для early stopping
    experiment_id='E4',
    early_stopping_patience=7
)

results.append({
    'experiment_id': 'E4',
    'model_summary': model_e4.get_summary(),
    'optimizer': 'Adam',
    'lr': OPTIMIZER_CONFIG['lr'],
    'weight_decay': 0.0,
    'epochs_trained': history_e4['epochs_trained'],
    'best_val_accuracy': best_val_acc_e4,
    'best_val_loss': min(history_e4['val_loss']) if history_e4['val_loss'] else None
})

# Сохраняем лучшую модель (E4).
if best_model_state_e4:
    torch.save(best_model_state_e4, ARTIFACTS_DIR / 'best_model.pt')
    print(f"Лучшая модель (E4) сохранена в {ARTIFACTS_DIR / 'best_model.pt'}")

In [ ]:
# curves_best.png.

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history_e4['train_loss'], label='Train Loss', linewidth=2)
plt.plot(history_e4['val_loss'], label='Val Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title(f'E4: Loss (Early Stopping) - Best Val Acc: {best_val_acc_e4:.4f}')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_e4['train_accuracy'], label='Train Accuracy', linewidth=2)
plt.plot(history_e4['val_accuracy'], label='Val Accuracy', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('E4: Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'curves_best.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"График сохранен в {FIGURES_DIR / 'curves_best.png'}")

### 3.2. Часть B (S09): LR, оптимизаторы.

In [ ]:
# 3.2. Часть B: LR, оптимизаторы, weight decay.

# Используем архитектуру, как в E4.
base_model_config = {
    'hidden_sizes': [1024, 512, 256],
    'dropout_rate': (0.3 if best_e2_e3['experiment_id'] == 'E2' else 0.0),
    'use_batchnorm': (best_e2_e3['experiment_id'] == 'E3')
}

print("\n" + "="*60)
print("Эксперимент O1: Слишком большой Learning Rate (Adam, lr=0.1)")
print("="*60)
model_o1 = MLP(**base_model_config).to(device)
optimizer_o1 = optim.Adam(model_o1.parameters(), lr=0.1)
history_o1, _, best_val_acc_o1 = run_experiment(
    model_o1, train_loader, val_loader, criterion, optimizer_o1, device,
    num_epochs=8, experiment_id='O1'
)
results.append({
    'experiment_id': 'O1',
    'model_summary': model_o1.get_summary(),
    'optimizer': 'Adam',
    'lr': 0.1,
    'momentum': 0.0,
    'weight_decay': 0.0,
    'epochs_trained': 8,
    'best_val_accuracy': best_val_acc_o1,
    'best_val_loss': min(history_o1['val_loss']) if history_o1['val_loss'] else None
})

print("\n" + "="*60)
print("Эксперимент O2: Слишком маленький Learning Rate (Adam, lr=1e-5)")
print("="*60)
model_o2 = MLP(**base_model_config).to(device)
optimizer_o2 = optim.Adam(model_o2.parameters(), lr=1e-5)
history_o2, _, best_val_acc_o2 = run_experiment(
    model_o2, train_loader, val_loader, criterion, optimizer_o2, device,
    num_epochs=8, experiment_id='O2'
)
results.append({
    'experiment_id': 'O2',
    'model_summary': model_o2.get_summary(),
    'optimizer': 'Adam',
    'lr': 1e-5,
    'momentum': 0.0,
    'weight_decay': 0.0,
    'epochs_trained': 8,
    'best_val_accuracy': best_val_acc_o2,
    'best_val_loss': min(history_o2['val_loss']) if history_o2['val_loss'] else None
})

print("\n" + "="*60)
print("Эксперимент O3: SGD + Momentum + Weight Decay")
print("="*60)
model_o3 = MLP(**base_model_config).to(device)
optimizer_o3 = optim.SGD(model_o3.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
history_o3, _, best_val_acc_o3 = run_experiment(
    model_o3, train_loader, val_loader, criterion, optimizer_o3, device,
    num_epochs=15, experiment_id='O3'
)
results.append({
    'experiment_id': 'O3',
    'model_summary': model_o3.get_summary(),
    'optimizer': 'SGD',
    'lr': 0.01,
    'momentum': 0.9,
    'weight_decay': 1e-4,
    'epochs_trained': 15,
    'best_val_accuracy': best_val_acc_o3,
    'best_val_loss': min(history_o3['val_loss']) if history_o3['val_loss'] else None
})

In [ ]:
# 4. Артефакты: curves_lr_extremes.png.

plt.figure(figsize=(15, 5))

# График Loss.
plt.subplot(1, 2, 1)
plt.plot(history_o1['val_loss'], label='O1 (LR=0.1) - слишком большой', 
         linewidth=2, marker='o', markersize=4)
plt.plot(history_o2['val_loss'], label='O2 (LR=1e-5) - слишком маленький', 
         linewidth=2, marker='s', markersize=4)
plt.plot(history_e4['val_loss'][:8], label='E4 (LR=0.001) - хороший', 
         linewidth=2, marker='^', markersize=4, alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Влияние Learning Rate на Val Loss')
plt.legend()
plt.grid(True)

# График Accuracy.
plt.subplot(1, 2, 2)
plt.plot(history_o1['val_accuracy'], label='O1 (LR=0.1) - слишком большой', 
         linewidth=2, marker='o', markersize=4)
plt.plot(history_o2['val_accuracy'], label='O2 (LR=1e-5) - слишком маленький', 
         linewidth=2, marker='s', markersize=4)
plt.plot(history_e4['val_accuracy'][:8], label='E4 (LR=0.001) - хороший', 
         linewidth=2, marker='^', markersize=4, alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.title('Влияние Learning Rate на Val Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'curves_lr_extremes.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"График сохранен в {FIGURES_DIR / 'curves_lr_extremes.png'}")

print("\n" + "="*60)
print("АНАЛИЗ LEARNING RATE:")
print("="*60)
print(f"O1 (LR=0.1): Финальная Val Acc = {best_val_acc_o1:.4f}")
print(f"  - Проблема: {'Loss расходится' if best_val_acc_o1 < 0.2 else 'Нестабильное обучение'}")
print(f"O2 (LR=1e-5): Финальная Val Acc = {best_val_acc_o2:.4f}")
print(f"  - Проблема: {'Слишком медленное обучение' if best_val_acc_o2 < 0.3 else 'Не успевает обучиться'}")
print(f"E4 (LR=0.001): Финальная Val Acc = {best_val_acc_e4:.4f}")
print(f"  - Хороший LR: стабильное обучение и хороший результат")

### Финальная оценка лучшей модели на тесте.

In [ ]:
# Загружаем сохраненное состояние лучшей модели.
model_e4.load_state_dict(torch.load(ARTIFACTS_DIR / 'best_model.pt'))

# Оцениваем на тесте.
test_loss, test_accuracy = evaluate(model_e4, test_loader, criterion, device)
print("="*60)
print("ФИНАЛЬНАЯ ОЦЕНКА ЛУЧШЕЙ МОДЕЛИ (E4)")
print("="*60)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Сохраняем результат
with open(ARTIFACTS_DIR / 'best_model_test_accuracy.txt', 'w') as f:
    f.write(f"Test Accuracy: {test_accuracy:.4f}\n")
    f.write(f"Test Loss: {test_loss:.4f}")

# Покажем примеры предсказаний.
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

model_e4.eval()
data_iter = iter(test_loader)
images, labels = next(data_iter)
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model_e4(images)
    _, predicted = torch.max(outputs, 1)

# Покажем несколько примеров.
plt.figure(figsize=(15, 8))
for i in range(10):
    plt.subplot(2, 5, i+1)
    img = images[i].cpu().numpy().transpose(1, 2, 0)
    # Денормализация.
    mean = np.array([0.4914, 0.4822, 0.4465])
    std = np.array([0.2470, 0.2435, 0.2616])
    img = std * img + mean
    img = np.clip(img, 0, 1)
    
    plt.imshow(img)
    true_label = cifar10_classes[labels[i].item()]
    pred_label = cifar10_classes[predicted[i].item()]
    color = 'green' if labels[i].item() == predicted[i].item() else 'red'
    plt.title(f'True: {true_label}\nPred: {pred_label}', color=color, fontsize=10)
    plt.axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'test_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4. Артефакты: runs.csv и best_config.json.

# Сохраняем runs.csv.
csv_filename = ARTIFACTS_DIR / 'runs.csv'
with open(csv_filename, mode='w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['experiment_id', 'dataset', 'seed', 'model_summary', 'optimizer',
                  'lr', 'momentum', 'weight_decay', 'epochs_trained',
                  'best_val_accuracy', 'best_val_loss']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for res in results:
        res['dataset'] = 'CIFAR10'
        res['seed'] = SEED
        res['momentum'] = res.get('momentum', 0.0)

        row = {field: res.get(field, '') for field in fieldnames}
        writer.writerow(row)

print(f"Результаты сохранены в {csv_filename}")

print("\n" + "="*60)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*60)
print(f"{'Exp':<5} {'Model':<30} {'Optimizer':<10} {'LR':<10} {'Epochs':<8} {'Val Acc':<10}")
print("-"*70)
for res in results:
    print(f"{res['experiment_id']:<5} {res['model_summary']:<30} "
          f"{res['optimizer']:<10} {res['lr']:<10} {res['epochs_trained']:<8} "
          f"{res['best_val_accuracy']:.4f}")

# Сохраняем best_config.json.
best_config = {
    'experiment_id': 'E4',
    'dataset': 'CIFAR10',
    'seed': SEED,
    'model_architecture': {
        'hidden_sizes': base_model_config['hidden_sizes'],
        'dropout_rate': base_model_config['dropout_rate'],
        'use_batchnorm': base_model_config['use_batchnorm'],
        'activation': 'relu',
        'total_parameters': sum(p.numel() for p in model_e4.parameters())
    },
    'training_params': {
        'optimizer': 'Adam',
        'lr': OPTIMIZER_CONFIG['lr'],
        'batch_size': BATCH_SIZE,
        'early_stopping_patience': 7,
        'max_epochs': 40,
        'actual_epochs_trained': history_e4['epochs_trained']
    },
    'final_test_metrics': {
        'accuracy': test_accuracy,
        'loss': test_loss
    },
    'best_val_accuracy': best_val_acc_e4
}

with open(ARTIFACTS_DIR / 'best_config.json', 'w', encoding='utf-8') as f:
    json.dump(best_config, f, indent=4)

print(f"\nКонфиг лучшей модели сохранен в {ARTIFACTS_DIR / 'best_config.json'}")

# Проверка всех созданных файлов.
print("\n" + "="*60)
print("СОЗДАННЫЕ АРТЕФАКТЫ:")
print("="*60)
for f in ARTIFACTS_DIR.glob('*'):
    print(f"  - {f.name}")
for f in FIGURES_DIR.glob('*'):
    print(f"  - figures/{f.name}")